# LLM Alignment

### <b> Datasets </b>
Dataset used to pre-train the already built LLM: https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu (not done by me, already provided trained)

Dataset used to align the LMM: https://huggingface.co/datasets/mlabonne/orpo-dpo-mix-40k (alignment done by me)

### <b> Informations </b>
To align we do not need as many data as to pre-train the LLM;

The chosen iteration dataset will give the content for the user role (human) and the answer for the asistent role (LLM);

The model will be trained seeing the right align and the wrong align to learn what (or not) to do

## Import Libraries

In [ ]:
# Import the necessary libraries
import os, sys
import math
from tqdm import tqdm
from datetime import datetime
import ipdb
from typing import List, Dict, Union

# Import pytorch libraries
import torch
import torch.nn as nn
from torch.nn import functional as F

# Import HugginFace libraries
import transformers
from datasets import load_dataset, load_from_disk

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.cuda.empty_cache()

# View the entire tensor
torch.set_printoptions(threshold=10000)

## Alignment Training Parameters

In [ ]:
batch_size = 1
epochs = 3
lr = 6e-5
lr_warmup_steps = 100 #in the first 100 iterations we will increase the lr and then aply the sheduler
context = 1024
alpha = 0.5 # scaling factor for the ORPO alignment technique odds ratio
prompt_max_size = 512 # limit for the prompt part of the iteraction 
compile = False # improve the performance of the pytorch calculation (true compiles faster but might not be compatible with some GPUs)
dtype = torch.bfloat16
log_iters = 200
eval_iters= 20

## Alignment Hyperparameters


In [ ]:
dropout=0
grad_clip=1.0
weight_decay=0.0

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Your device is:",device)

## Alignment Logging

In [ ]:
wandb_log = True
wandb_project = "my_llm_aligned"
wandb_run_name = "my_llm_aligned-" + datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

if wandb_log:
    import wandb
    wandb.init(project=wandb_project, name=wandb_run_name)

## Load and Tokenizing the Alignment Dataset

https://huggingface.co/datasets/mlabonne/orpo-dpo-mix-40k

In [ ]:
dataset_path = "./data/orpo_dataset2"
dataset_name = "mlabonne/orpo-dpo-mix-40k"
tokenizer_path = "tokenizers/tok16384"
checkpoint_dir = "./models"

# Tokenizing dataset
tokenizer = transformers.AutoTokenizer.from_pretrained(tokenizer_path) # load the tokenizer in HugginsFace format to use the same template

# Set the interaction template
tokenizer.chat_template = "{% for message in messages %}{% if message['role'] =='user' %}\n{{'<|user|>\n' + message['content'] + eos_token}}\n{% elif message['role'] =='assistant' %}\n{{'<|assistant|>\n' + message['content'] + eos_token}}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{'<|assistant|>'}}\n{%endif%}\n{% endfor%}"

# Make padding token equal to the end of sentence token (ID of 2 in our case)
tokenizer.pad_token = tokenizer.eos_token


# Preparing the Alignment training dataset
# 1 -> Filter the dataset (Bad topics remove and sequences higher than context window removed)
# 2 -> Pre-Process and Tokenize the data
if os.path.exists(dataset_path):
    
    print("loading already tokenized dataset")
    dataset = load_from_disk(dataset_path)
    
else:
    # Load the dataset
    print("Filtering and Tokenizing the dataset")
    dataset = load_dataset(dataset_name, split = 'all')

    # 1 - Filter out BAD topics: remove contorversial topics from being aligned) - Introducing guardrails in the LLM
    dataset = dataset.filter(lambda r: r['source'] != "toxic-dpo-v0.2")

    # 1- Filter out LONG sequences: remove sequences to fit in the total context of the LLM (prompt + answer < context)
    def filter_dataset(examples):
        # Get off the last answer because that is what the llm should answer
        prompt_length = tokenizer.apply_chat_template(examples['chosen'][:-1], tokenize = True, add_generation_prompt=True, return_tensors='pt', return_dict=False).size(-1) 
        if prompt_length < prompt_max_size:
            return True
        else:
            return False
            
    dataset = dataset.filter(filter_dataset)

    # 2- Pre-process and tokenize the data
    def preprocess_dataset(examples: Union[List,Dict]):
        # take the field, eliminate last answer, apply the chat template and add assistance prompt
        prompt = [tokenizer.apply_chat_template(item[:-1], tokenize = False, add_generation_prompt = True) for item in examples['chosen']]
        chosen = [tokenizer.apply_chat_template(item, tokenize=False) for item in examples['chosen']]
        rejected = [tokenizer.apply_chat_template(item, tokenize=False) for item in examples['rejected']]

        # tokenize
        # fileds: ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing
        inputs = tokenizer(prompt, max_length=context, padding="max_length", truncation = True, return_tensor='pt')
        pos_labels = tokenizer(chosen, max_length=context, padding="max_length", truncation = True, return_tensor='pt')
        neg_labels = tokenizer(rejected, max_length=context, padding="max_length", truncation = True, return_tensor='pt')

        inputs['positive_input_ids'] = pos_labels['input_ids']
        inputs['positive_attention_mask'] = pos_labels['attention_mask']

        inputs['negative_input_ids'] = neg_labels['input_ids']
        inputs['negative_attention_mask'] = neg_labels['attention_mask']

        return inputs


    # 2 - Pre-Process and tokenize dataset (by default it send batches of 1000)
    dataset = dataset.map(preprocess_dataset, batched=True, remove_columns=dataset.column_names)

    # Save the dataset in the disk
    dataset.save_to_disk(dataset_path)

## Split the Alignment Dataset (Train and Validation)

In [ ]:
dataset = dataset.shuffle(42).train_test_split(test_size=0.05)

# Features: input_ids, attention_mask
train_data = dataset['train']
val_data = dataset['test']

# Prepare data for language modeling
data_collator = transformers.DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm=False)

# Setup data loaders (help request batches from data)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle = False, collate_fn = data_collator)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=batch_size, shuffle = False, collate_fn = data_collator)

val_iterator=iter(val_loader)
train_iterator=iter(train_loader)

## Setup Alignment Architecture

In [ ]:
# Import the pre-trained LLM
from llm import Llama, ModelArgs

checkpoint = torch.load(os.path.join(checkpoint_dir, "base_model.pt"), weights_only=False)
config = checkpoint.pop("config")

# Load the LLM model with the checkpoint parameters that are stored in the checkpoint
model_args = ModelArgs(
    dim=config.hidden_size,
    n_layers=config.num_hidden_layers,
    n_heads=config.num_attention_heads, 
    n_kv_heads=config.num_key_value_heads,
    vocab_size=config.vocab_size,
    norm_eps=config.rms_norm_eps,
    rope_theta=config.rope_theta,
    max_seq_len=context,
    dropout=config.attention_dropout,
    hidden_dim=config.intermediate_size,
    attention_bias=config.attention_bias,
    mlp_bias=config.mlp_bias    
)


# Instantiate the model and load the dictiuonary state from the checkpoint
model = Llama(model_args)
model.load_state_dict(checkpoint)
model = model.to(dtype)
model = model.to(device)
model.train()

## Setup Alignment Training Parameters

In [ ]:
# Print number of parameters (GPT 3 - 175 Billions parameters)
print(sum(p.numel() for p in model.parameters()) / 1e6, "Million parameters")

# Setup the optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.99), eps=1e-8, fused=device=='cuda', weight_decay=weight_decay)

# training steps declaration
num_training_steps = len(train_loader) * epochs

# Setup the Scheduler for changing the learning rate

def lr_lambda(current_step):
    if current_step < lr_warmup_steps:
        return float(current_step) / float(max(1,lr_warmup_steps))
    progress = float(current_step - lr_warmup_steps) / float(max(1,num_training_steps-lr_warmup_steps))
    return max(0.0, 0.5*(1*math.cos(math.pi*float(0.5) *2.0 * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda, last_epoch=-1)

## Alignment Training Loop

Using the ORPO technique for the alignment loss: https://arxiv.org/html/2403.07691

In [ ]:
# A loss that pushes the network to produce answers with the positive view rather than the negative during the training phase
def compute_logps(prompt_attention_mask, chosen_inputs, chosen_attention_mask, logits):

    # The mask will have 1 in the last prompt token and 1s in all the answer tokens
    mask = chosen_attention_mask[:,:-1]-prompt_attention_mask[:,1:] 

    # (mask*chosen_inputs[:,1:] -> Isolate the answer tokens (everything 0s excpet the answer)
    # the unsqueeze adds an extra dimension to match the indexes and the logits dimensions
    # It will extract the predictions of the model (logits) for the indexs of the answer we isoltaed previously
    per_token_logps = torch.gather(logits[:,:-1,:].log_softmax(-1), dim=2, index=(mask*chosen_inputs[:,1:]).unsqueeze(2)).squeeze(2)

    # Get the average probabilities
    return torch.mul(per_token_logps, mask.to(dtype=torch.bfloat16)).sum(dim=1).to(dtype=torch.float64)/mask.sum(dim=1).to(dtype)

In [ ]:
@torch.no_grad()  # Prevent gradient calculation
# Calculate average of training and validation losses over multiple batches
def calculate_loss():
    global train_iterator, val_iterator
    loss_mean={}
    odds_mean={}
    ratio_mean={}
    model.eval()
    for split in ['train','val']: 
        l=torch.zeros(eval_iters)  # Create a tensor of zeros the size of eval_iters
        o=torch.zeros(eval_iters)  # Create a tensor of zeros the size of eval_iters
        r=torch.zeros(eval_iters)  # Create a tensor of zeros the size of eval_iters
        for i in range(eval_iters):
            try:
                if split == 'val':
                    batch = next(val_iterator)
                else:
                    batch = next(train_iterator)
            except StopIteration:
                if split == 'val':
                    print("####### Resetting Validation Iterator")
                    val_iterator = iter(val_loader)
                    batch = next(val_iterator)
                else:
                    print("####### Resetting Training Iterator")
                    train_iterator = iter(train_loader)
                    batch = next(train_iterator)                   

            batch["positive_input_ids"] = batch["positive_input_ids"].to(device) 
            batch["positive_attention_mask"] = batch["positive_attention_mask"].to(device)
            batch["negative_input_ids"] = batch["negative_input_ids"].to(device)
            batch["negative_attention_mask"] = batch["negative_attention_mask"].to(device)
            batch["attention_mask"] = batch["attention_mask"].to(device)

            # clone the information to another variable
            neg_labels = batch['negative_input_ids'].clone()
            pos_labels = batch['positive_input_ids'].clone()
        
            mask = batch['attention_mask'] * batch['positive_attention_mask']  # sets mask to have 1s in only the prompt positions
            pos_labels = pos_labels * mask.logical_not()  # puts 0s where the prompt was, preserve last answer (padding tokens are EOS(2))
        
            pos_labels[pos_labels == 0] = tokenizer.pad_token_id # replaces 0s with EOS(2)
            neg_labels[neg_labels == tokenizer.pad_token_id] = -100 # change 2 to -100 so that loss calculations ignore prompt and padding
            pos_labels[pos_labels == tokenizer.pad_token_id] = -100 # change 2 to -100 so that loss calculations ignore prompt and padding
        
            outputs_pos, loss_pos = model(batch['positive_input_ids'], pos_labels)  #  (1,1024) , (1,1024)
            outputs_neg, loss_neg = model(batch['negative_input_ids'], neg_labels)    
        
            # calculate per token log probabilities, essential to calculate the ORPO LOG ODDS RATIO (prefere positive answers than negative ones)
            # 1) compute log probabilities
            pos_prob = compute_logps(
                        prompt_attention_mask=batch['attention_mask'], 
                        chosen_inputs=batch["positive_input_ids"], 
                        chosen_attention_mask=batch['positive_attention_mask'], 
                        logits=outputs_pos
                    )
            # returns the average of the log probabilities for the negative samples (masking out prompt)
            neg_prob = compute_logps(
                        prompt_attention_mask=batch['attention_mask'], 
                        chosen_inputs=batch["negative_input_ids"], 
                        chosen_attention_mask=batch['negative_attention_mask'], 
                        logits=outputs_neg
                    )    
        
            # CALCULATE ORPO ODDS RATIO
            log_odds = (pos_prob - neg_prob) - (torch.log(1 - torch.exp(pos_prob)) - torch.log(1 - torch.exp(neg_prob))) # The ORPO loss computation
            sig_ratio = F.sigmoid(log_odds) # constrain to be between 0 and 1
            ratio = torch.log(sig_ratio) # apply the final log to the calculation
        
            # Calculate the Final Total Loss, combination of standard Cross Entropy loss and the weighted Odds Ratio
            loss = torch.mean(loss_pos - (alpha*ratio).mean()).to(dtype=dtype)

            l[i]=loss.item()
            o[i]=log_odds.mean().item()
            r[i]=ratio.mean().item()
        
        loss_mean[split]=l.mean().item()
        odds_mean[split]=o.mean().item()
        ratio_mean[split]=r.mean().item()
        
            
    model.train()
    return loss_mean, odds_mean, ratio_mean

l, o, r = calculate_loss()
print(l,o,r)

In [ ]:
try:
    for e in range(epochs):
        for i, batch in tqdm(enumerate(train_loader), total=len(train_loader), dynamic_ncols=True):
            
            optimizer.zero_grad(set_to_none = True) # Compute new gradients

            # Move batch data to device
            batch["positive_input_ids"] = batch["positive_input_ids"].to(device)
            batch["positive_attention_mask"] = batch["positive_attention_mask"].to(device)
            batch["negative_input_ids"] = batch["negative_input_ids"].to(device)
            batch["negative_attention_mask"] = batch["negative_attention_mask"].to(device)
            batch["attention_mask"] = batch["attention_mask"].to(device)

            # clone the information to another variable
            neg_labels = batch['negative_input_ids'].clone()
            pos_labels = batch['positive_input_ids'].clone()

            mask = batch['attention_mask'] * batch['positive_attention_mask']  # sets mask to have 1s in only the prompt positions
            pos_labels = pos_labels * mask.logical_not()  # puts 0s where the prompt was, preserve last answer (padding tokens are EOS(2))

            pos_labels[pos_labels == 0] = tokenizer.pad_token_id # replaces 0s with EOS(2)
            neg_labels[neg_labels == tokenizer.pad_token_id] = -100 # change 2 to -100 so that loss calculations ignore prompt and padding
            pos_labels[pos_labels == tokenizer.pad_token_id] = -100 # change 2 to -100 so that loss calculations ignore prompt and padding

            outputs_pos, loss_pos = model(batch['positive_input_ids'], pos_labels)  #  (1,1024) , (1,1024)
            outputs_neg, loss_neg = model(batch['negative_input_ids'], neg_labels)

            # calculate per token log probabilities, essential to calculate the ORPO LOG ODDS RATIO (prefere positive answers than negative ones)
            pos_prob = compute_logps(
                        prompt_attention_mask=batch['attention_mask'], 
                        chosen_inputs=batch["positive_input_ids"], 
                        chosen_attention_mask=batch['positive_attention_mask'], 
                        logits=outputs_pos
                    )
            neg_prob = compute_logps(
                        prompt_attention_mask=batch['attention_mask'], 
                        chosen_inputs=batch["negative_input_ids"], 
                        chosen_attention_mask=batch['negative_attention_mask'], 
                        logits=outputs_neg
                    )

            # CALCULATE ORPO ODDS RATIO
            log_odds = (pos_prob - neg_prob) - (torch.log(1 - torch.exp(pos_prob)) - torch.log(1 - torch.exp(neg_prob)))
            sig_ratio = F.sigmoid(log_odds) # constrain to be between 0 and 1
            ratio = torch.log(sig_ratio) # apply the final log to the calculation

            # Calculate the Final Total Loss, combination of standard Cross Entropy loss and the weighted Odds Ratio
            loss = torch.mean(loss_pos - (alpha*ratio).mean()).to(dtype=dtype)

            if i%log_iters==0:
                
                # Call the Loss function (normal loss + ORPO loss)
                loss_m, log_odds_m, ration_m = calculate_loss()

                print(f"Epochs: [{e}/{epochs}] Step: [{i}/{len(train_loader)}], train loss: {loss_m['train']:.3f}, val loss: {loss_m['val']:.3f}, Train Odds Ratio: {log_odds_m['train']:.3f}, Val Odds Ratio: {log_odds_m['val']:.3f}")

                if wandb_log:
                    wandb.log({
                        "loss/train": loss_m['train'],
                        "loss/val": loss_m['val'],
                        "log_odds/train": log_odds_m['train'],
                        "log_odds/val": log_odds_m['val'],
                        "lr": scheduler.get_last_lr()[0],
                    },
                    step = (e*len(train_loader)+i))

            loss.backward() # propagate the loss
            nn.utils.clip_grad_norm_(model.parameters(), max_norm = grad_clip)
            optimizer.step()
            scheduler.step()

    # Save checkpoints epoch by epoch
    sd = model.state_dict()
    sd['config'] = config
    torch.save(sd, os.path.join(checkpoint_dir, f'{project_name}_{e+1}.pt'))
                              
    if wandb_log:
        wandb.finish()
        
except KeyboardInterrupt:
    print("Training Interrupted. Cleaning up ...")
    
finally:
    # Release GPU memory
    torch.cuda.empty_cache()
    print("GPU memory released!")